### 선형 회귀 실습 문제

# Tips Dataset
: 레스토랑의 총 식사 금액을 이용해 손님이 지불할 팁 금액을 예측하는 선형 회귀 모델 구현.

#### Step1. 데이터 불러오기 및 전처리

**사용 데이터: Tips Dataset**

- 데이터: 244개
- 입력 특징: 1개 (총 식사 금액 (`total_bill`))
- 정답: 팁 금액(`tip`)

| 열 이름 | 의미 | 설명 |
|:---:|:---:|:---|
| `total_bill` | 총 식사 금액 | 식사에 청구된 총 금액 |
| `tip` | 팁 금액 | 종업원에게 지불한 팁 |
| `sex` | 성별 | 계산한 손님의 성별 |
| `smoker` | 흡연 여부 | 흡연자 여부 |
| `day` | 요일 | 식사한 요일 |
| `time` | 식사 시간 | Lunch 또는 Dinner |
| `size` | 인원수 | 테이블의 손님 수 |

**실습 조건**
- 입력값(X): `total_bill`
- 정답(y): `tip`
- 결측치가 있는지 확인하고, 필요한 경우 처리한다.
- 학습 데이터와 테스트 데이터를 8:2 비율로 분리한다.
- 입력값은 학습 데이터의 평균과 표준편차를 이용해 표준화한다.
- `total_bill`과 `tip`의 관계를 산점도로 먼저 확인한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# 재현 가능한 실험을 위한 시드 설정
torch.manual_seed(42)
np.random.seed(42)

# Tips 데이터 불러오기
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
df = pd.read_csv(url)

# 이번 실습에 필요한 두 열만 선택하고 결측치 제거
df = df[["total_bill", "tip"]].dropna().copy()

print(df.head())
print("데이터 크기:", df.shape)
print("결측치 개수:\n", df.isnull().sum())

# 두 변수의 관계 확인
plt.figure(figsize=(6, 4))
plt.scatter(df["total_bill"], df["tip"], alpha=0.7)
plt.xlabel("Total bill")
plt.ylabel("Tip")
plt.title("Total bill vs Tip")
plt.grid(alpha=0.3)
plt.show()

# 입력(X), 정답(y) 분리
X = df[["total_bill"]].values.astype(np.float32)
y = df[["tip"]].values.astype(np.float32)

# 학습 데이터와 테스트 데이터를 8:2로 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 학습 데이터의 평균과 표준편차로 입력값 표준화
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)
X_std[X_std == 0] = 1.0

X_train_scaled = (X_train - X_mean) / X_std
X_test_scaled = (X_test - X_mean) / X_std

# NumPy 배열을 PyTorch 텐서로 변환
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

print("X_train:", X_train_t.shape)
print("X_test:", X_test_t.shape)
print("y_train:", y_train_t.shape)
print("y_test:", y_test_t.shape)

#### Step2. 모델 만들기

- 입력 노드: 1개 (`total_bill`)
- 출력 노드: 1개 (`tip`)
- 활성화 함수: 사용하지 않음
- 모델: `nn.Linear`

In [ ]:
# 입력 1개(total_bill), 출력 1개(tip)의 선형 회귀 모델
model = nn.Linear(in_features=1, out_features=1)

print(model)
print("초기 가중치:", model.weight)
print("초기 편향:", model.bias)

#### STEP 3. 모델 학습시키기

- Optimizer: SGD
- Loss function: MSELoss
- 학습률(LR): 0.01부터 시작
- Epoch: 200
- Batch size: 32

In [ ]:
# Dataset과 DataLoader 생성
BATCH_SIZE = 32

train_DS = TensorDataset(X_train_t, y_train_t)
train_DL = DataLoader(train_DS, batch_size=BATCH_SIZE, shuffle=True)

# 손실 함수와 옵티마이저
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 모델 학습
EPOCHS = 200
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_DL:
        # 1) 예측
        pred = model(X_batch)

        # 2) 손실 계산
        loss = criterion(pred, y_batch)

        # 3) 이전 기울기 초기화
        optimizer.zero_grad()

        # 4) 기울기 계산
        loss.backward()

        # 5) 파라미터 업데이트
        optimizer.step()

        # 배치별 평균 손실을 구하기 위해 샘플 수를 반영
        epoch_loss += loss.item() * X_batch.size(0)

    epoch_loss /= len(train_DS)
    loss_history.append(epoch_loss)

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch + 1:3d}/{EPOCHS}], Loss: {epoch_loss:.4f}")

# 학습 손실 변화 확인
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.grid(alpha=0.3)
plt.show()

#### Step4. 모델 테스트하기


In [ ]:
# 테스트 데이터로 예측
model.eval()
with torch.no_grad():
    y_pred_t = model(X_test_t)

# 평가 지표 계산
test_loss = criterion(y_pred_t, y_test_t).item()
mae = torch.mean(torch.abs(y_pred_t - y_test_t)).item()

# R^2 직접 계산
ss_res = torch.sum((y_test_t - y_pred_t) ** 2)
ss_tot = torch.sum((y_test_t - torch.mean(y_test_t)) ** 2)
r2 = (1 - ss_res / ss_tot).item()

print(f"Test MSE: {test_loss:.4f}")
print(f"Test MAE: {mae:.4f}")
print(f"Test R²: {r2:.4f}")

# 실제값과 예측값 일부 비교
for actual, predicted in zip(y_test_t[:10], y_pred_t[:10]):
    print(f"실제 팁: {actual.item():.2f}, 예측 팁: {predicted.item():.2f}")

#### Step5. 테스트 결과 시각화

**문제**
- 첫 번째 그래프: 테스트 데이터의 실제 산점도와 모델의 예측 회귀선을 함께 표시.
- 두 번째 그래프: 실제 팁 금액을 x축, 예측 팁 금액을 y축으로 하는 산점도를 작성.
- 두 번째 그래프에 완벽한 예측을 나타내는 기준선(y=x)을 추가.
- 그래프의 축 이름과 제목을 적절히 표시.

In [ ]:
# 그래프를 위해 원래 단위의 total_bill과 예측값 사용
X_test_original = X_test.reshape(-1)
y_test_original = y_test.reshape(-1)
y_pred_original = y_pred_t.numpy().reshape(-1)

# 첫 번째 그래프: 실제 테스트 데이터 + 회귀선
order = np.argsort(X_test_original)

plt.figure(figsize=(7, 5))
plt.scatter(X_test_original, y_test_original, label="Actual data", alpha=0.7)
plt.plot(
    X_test_original[order],
    y_pred_original[order],
    color="red",
    label="Predicted regression line"
)
plt.xlabel("Total bill")
plt.ylabel("Tip")
plt.title("Actual Tips and Predicted Regression Line")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 두 번째 그래프: 실제 팁과 예측 팁 비교
plt.figure(figsize=(6, 6))
plt.scatter(y_test_original, y_pred_original, alpha=0.7)

# 완벽한 예측을 의미하는 y=x 기준선
min_value = min(y_test_original.min(), y_pred_original.min())
max_value = max(y_test_original.max(), y_pred_original.max())
plt.plot([min_value, max_value], [min_value, max_value],
         "r--", label="Perfect prediction (y=x)")

plt.xlabel("Actual tip")
plt.ylabel("Predicted tip")
plt.title("Actual vs Predicted Tip")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

#### STEP 6. 파라미터 개수 확인

In [ ]:
# nn.Linear(1, 1)의 학습 가능한 파라미터 개수
total_params = sum(param.numel() for param in model.parameters())
trainable_params = sum(
    param.numel() for param in model.parameters() if param.requires_grad
)

print("가중치(weight) 개수:", model.weight.numel())
print("편향(bias) 개수:", model.bias.numel())
print("전체 파라미터 개수:", total_params)
print("학습 가능한 파라미터 개수:", trainable_params)